In [1]:
import os
import pathlib

import json
from tqdm.auto import tqdm

import requests

In [2]:
target_db = pathlib.Path('./public/database/')

In [3]:
# with open(target_db / 'root_data.json') as f:
#     root_data = json.load(f)

# root_data_filtered = []
# for crag in root_data:
#     if crag['bought']:
#         root_data_filtered.append(crag)

# Crawl API

In [21]:
with open('token.txt') as f:
    token = f.read().strip()

In [22]:
header = {'authorization': f"Token {token}"}

In [23]:
root_url = "https://www.versantesud.it/api/keinwunder/climbing/sport/"

In [24]:
response = requests.get(root_url ,headers=header)

In [25]:
response.ok

True

In [26]:
root_data = response.json()

with open(target_db / 'root_data.json', 'w') as f:
    json.dump(root_data, f)

In [27]:
root_data_filtered = []
for crag in root_data:
    if crag['bought']:
        root_data_filtered.append(crag)

In [28]:
with open(target_db / 'root_data_bought.json', 'w') as f:
    json.dump(root_data_filtered, f)

In [29]:
# Purchased guides
url = "https://www.versantesud.it/api/keinwunder/climbing/purchased/guide/"
response = requests.get(url ,headers=header)
assert response.ok
purchased_guide_data = response.json()
with open(target_db / 'purchased_guides.json', 'w') as f:
    json.dump(purchased_guide_data, f)

In [30]:
with open(target_db / 'purchased_guides.json') as f:
    purchased_guide_data = json.load(f)

In [31]:
# Book cover images
for guide in tqdm(purchased_guide_data):
    guide_id = f"{guide['title']}_{guide['year']}"
    file_name = guide['cover_image'].split('/')[-1]
    
    output_file = target_db / 'images_book_covers' / file_name
    
    if output_file.is_file():
        continue
    
    
    response = requests.get(guide['cover_image'])
    assert response.ok
    
    with open(output_file, 'wb') as f:
        f.write(response.content)
    

  0%|          | 0/8 [00:00<?, ?it/s]

In [32]:
# Download crags
for crag in tqdm(root_data_filtered):
    if not crag['bought']:
        continue
    
    assert crag['url'].startswith(root_url)
    assert crag['url'].endswith('/')
    crag_id = crag['url'][len(root_url):-1]
    
    output_path = target_db / f'crags/{crag_id}.json'
    
    if output_path.is_file():
        continue
    
    response = requests.get(crag['url'], headers=header)
    assert response.ok
    
    crag_data = response.json()
    
    with open(output_path, 'w') as f:
        json.dump(crag_data, f)
    

  0%|          | 0/702 [00:00<?, ?it/s]

In [33]:
crag_ids_bought = []
for crag in tqdm(root_data):
    if not crag['bought']:
        continue
    
    assert crag['url'].startswith(root_url)
    assert crag['url'].endswith('/')
    crag_id = crag['url'][len(root_url):-1]
    
    crag_ids_bought.append(crag_id)

  0%|          | 0/1749 [00:00<?, ?it/s]

In [34]:
len(crag_ids_bought)

702

In [35]:
# Download approaches and sectors (list of sectors) for each crag
for crag_id in tqdm(crag_ids_bought):
    with open(target_db / f'crags/{crag_id}.json') as f:
        crag_data = json.load(f)
        
        
    assert crag_data['approach_url'] == root_url + crag_id + '/approach/'
    assert crag_data['sectors_url'] == root_url + crag_id + '/sectors/'
    
    output_path_approach = target_db / f'approaches/{crag_id}.json'
    output_path_crag = target_db / f'sectors/{crag_id}.json'
    
        
    if not output_path_approach.is_file():
        response = requests.get(crag_data['approach_url'], headers=header)
        assert response.ok
        approach_data = response.json()
        with open(output_path_approach, 'w') as f:
            json.dump(approach_data, f)
    
    
    if not output_path_crag.is_file():
        response = requests.get(crag_data['sectors_url'], headers=header)
        assert response.ok
        sectors_data = response.json()
        with open(output_path_crag, 'w') as f:
            json.dump(sectors_data, f)

  0%|          | 0/702 [00:00<?, ?it/s]

In [36]:
# download sector instances
for crag_id in tqdm(crag_ids_bought):
    with open(target_db / f'sectors/{crag_id}.json') as f:
        sector_data = json.load(f)
        
    for sector in sector_data['sectors']:
        sector_id = sector['url'].split('/')[-2]
        assert sector['url'] == root_url + crag_id + '/sector/' + sector_id + '/'
        
        output_path = target_db / f'sector_instance/{crag_id}_{sector_id}.json'
        if output_path.is_file():
            continue
        
        response = requests.get(sector['url'], headers=header)
        assert response.ok
        
        sector_data = response.json()
        with open(output_path, 'w') as f:
            json.dump(sector_data, f)

  0%|          | 0/702 [00:00<?, ?it/s]

In [37]:
# download topo images
for crag_id in tqdm(crag_ids_bought):
    with open(target_db / f'sectors/{crag_id}.json') as f:
        sector_data = json.load(f)
        
    for sector in sector_data['sectors']:
        sector_id = sector['url'].split('/')[-2]
        
        with open(target_db / f'sector_instance/{crag_id}_{sector_id}.json') as f:
            sector_instance_data = json.load(f)
            
        for image in sector_instance_data['images']:
            image_filename = image['url'].split('/')[-1]
            output_file = target_db / 'images_topos' / image_filename
            
            if output_file.is_file():
                continue
            
            response = requests.get(image['url'])
            assert response.ok
            
            with open(output_file, 'wb') as f:
                f.write(response.content)

  0%|          | 0/702 [00:00<?, ?it/s]

In [38]:
# Download approach images
for crag_id in tqdm(crag_ids_bought):
    with open(target_db / f'approaches/{crag_id}.json') as f:
        approach_data = json.load(f)
        
    if approach_data['image'] is None:
        print('No approach image for', crag_id)
        continue
    
    image_filename = approach_data['image'].split('/')[-1]
    
    output_file = target_db / f'images_approaches/{image_filename}'
    if output_file.is_file():
        continue
    
    
    response = requests.get(approach_data['image'])
    assert response.ok
    
    with open(output_file, 'wb') as f:
        f.write(response.content)
        

  0%|          | 0/702 [00:00<?, ?it/s]

No approach image for 240
No approach image for 327
No approach image for 277
No approach image for 306


# Downsample images

In [39]:
from PIL import Image

In [40]:
for cat in ['approaches', 'topos']:
    max_size = 500 if cat == 'approaches' else 800

    path_src = target_db / f'images_{cat}'
    path_dst = target_db / f'images_optimized_{cat}'
    path_dst.mkdir(exist_ok=True)

    for img_path in tqdm(list(path_src.iterdir())):
        assert img_path.name.endswith('.png') or img_path.name.endswith('.jpg')
        
        im = Image.open(img_path).convert('RGB')
        im.thumbnail((max_size, max_size), )
        
        
        
        img_path_dst = path_dst / (img_path.name + '.jpg')
        im.save(img_path_dst, 'JPEG', optimize=True, quality=75)
        
        

  0%|          | 0/443 [00:00<?, ?it/s]

  0%|          | 0/1377 [00:00<?, ?it/s]

# Parse Isar

In [2]:
with open("offline_spots_v1.isar", "rb") as f:
    data = f.read()

In [42]:
i = 0
j = 0
datas = []
while i != -1:
    i = data.find(b'{', j)
    j = data.find(b'\x00', i)
    
    data_str = data[i:j-1]
    
    data_str = ''.join(chr(x) for x in data_str if x in range(128))
    
    if not data_str:
        continue
    
    data_parsed = json.loads(data_str)
    print(data_parsed)
    print('===')
    
    datas.append(data_parsed)

{'name': 'OfflineSpot', 'embedded': False, 'properties': [{'name': 'apiVersion', 'type': 'String', 'target': None}, {'name': 'approachImagePath', 'type': 'String', 'target': None}, {'name': 'approachJson', 'type': 'String', 'target': None}, {'name': 'boulderDetailsJson', 'type': 'String', 'target': None}, {'name': 'coverImagePath', 'type': 'String', 'target': None}, {'name': 'discipline', 'type': 'Byte', 'target': None}, {'name': 'displayName', 'type': 'String', 'target': None}, {'name': 'downloadedAt', 'type': 'Long', 'target': None}, {'name': 'formattedSize', 'type': 'String', 'target': None}, {'name': 'galleryImagePaths', 'type': 'StringList', 'target': None}, {'name': 'guideCoverImagePath', 'type': 'String', 'target': None}, {'name': 'hasTileRegions', 'type': 'Bool', 'target': None}, {'name': 'isValid', 'type': 'Bool', 'target': None}, {'name': 'latitude', 'type': 'Double', 'target': None}, {'name': 'longitude', 'type': 'Double', 'target': None}, {'name': 'mapSnapshotPath', 'type':

In [53]:
datas[-1]

{'name': 'FALESIA DELLORSO',
 'histogram': [{'label': '<5', 'count': 0},
  {'label': '5a', 'count': 0},
  {'label': '5b', 'count': 0},
  {'label': '5c', 'count': 0},
  {'label': '6a', 'count': 2},
  {'label': '6b', 'count': 2},
  {'label': '6c', 'count': 2},
  {'label': '7a', 'count': 2},
  {'label': '7b', 'count': 1},
  {'label': '7c', 'count': 1},
  {'label': '8a', 'count': 2},
  {'label': '8b', 'count': 0},
  {'label': '8c', 'count': 0},
  {'label': '9', 'count': 0},
  {'label': '?', 'count': 2}],
 'characteristics': ['Base confortevole',
  'Disponibilit parcheggio',
  'Splendido',
  'Ottima chiodatura',
  'Tranquillo / non affollato',
  'Calcare'],
 'cover_image': 'https://pesce-production-962565294008-eu-north-1-an.s3.amazonaws.com/filer_public_thumbnails/filer_public/ac/76/ac76ae84-1f07-43ad-a614-0c9fad60a177/immagine_cop_trentoevalsuganafalesie.jpg__4096x3072_q85_ALIAS-iclimbing_fullscreen_subsampling-2.jpg',
 'cover_characteristics': [{'name': 'Avvicinamento', 'value': '15 min'

In [52]:
json.loads(list(datas[-3].values())[0])['routes']

[{'number': 1,
  'name': 'SENZA NOME',
  'beautiful': True,
  'pitches': [{'name': 'L1',
    'length': None,
    'comment': '',
    'difficulty_grade': None,
    'done': False,
    'toggle_done_url': 'https://www.versantesud.it/api/keinwunder/climbing/sport/3280/sector/5664/route/55344/pitch/59206/toggle_done/'}]},
 {'number': 2,
  'name': 'PENTECOSTE',
  'beautiful': False,
  'pitches': [{'name': 'L1',
    'length': 30,
    'comment': 'Duro a met',
    'difficulty_grade': '7b',
    'done': False,
    'toggle_done_url': 'https://www.versantesud.it/api/keinwunder/climbing/sport/3280/sector/5664/route/55345/pitch/59207/toggle_done/'}]},
 {'number': 3,
  'name': 'SENZA NOME',
  'beautiful': False,
  'pitches': [{'name': 'L1',
    'length': 30,
    'comment': 'Continuit',
    'difficulty_grade': '6c+',
    'done': False,
    'toggle_done_url': 'https://www.versantesud.it/api/keinwunder/climbing/sport/3280/sector/5664/route/55346/pitch/59208/toggle_done/'}]},
 {'number': 4,
  'name': 'SENZA